# Praktikum Kriptografi RSA – Pertemuan 11

**Nama:** Bima Satria  
**NIM:** 24343052

Notebook ini mengikuti urutan jobsheet: pembangkitan kunci RSA, enkripsi/dekripsi numerik, beberapa pesan, repeated squaring, RSA-OAEP, RSA-AES, dan tiga soal evaluasi.

> **Catatan:** Contoh RSA numerik dengan bilangan prima kecil hanya untuk pembelajaran. Untuk penggunaan nyata, gunakan pustaka kriptografi, ukuran kunci yang memadai, padding yang aman, dan pengelolaan kunci yang benar.


## Persiapan pustaka
Jalankan sel berikut jika pustaka `cryptography` belum tersedia pada lingkungan Jupyter.

In [1]:
# Jika belum terpasang, hapus tanda komentar pada baris berikut:
# %pip install cryptography

from math import gcd


## Fungsi bantu: pembentukan kunci RSA sederhana
Fungsi ini digunakan pada bagian praktikum numerik dan evaluasi. Nilai `d` dihitung sebagai invers modular dari `e` terhadap φ(n).

In [2]:
def buat_kunci_rsa(p, q, e):
    """Membentuk parameter RSA sederhana dari dua bilangan prima p dan q."""
    if p == q:
        raise ValueError("p dan q harus merupakan bilangan prima yang berbeda.")

    n = p * q
    phi = (p - 1) * (q - 1)

    if gcd(e, phi) != 1:
        raise ValueError("e harus relatif prima terhadap φ(n).")

    d = pow(e, -1, phi)
    return n, phi, d


## D. Praktikum

### 1. Kunci Publik dan Kunci Privat

Menggunakan parameter pada jobsheet: `p = 5`, `q = 11`, dan `e = 3`. Modulus dihitung dengan `n = p × q`, sedangkan φ(n) dihitung dengan `(p−1)(q−1)`.


In [3]:
from math import gcd

p = 5
q = 11
e = 3

n, phi, d = buat_kunci_rsa(p, q, e)

print("p =", p)
print("q =", q)
print("n =", n)
print("phi(n) =", phi)
print("e =", e)
print("d =", d)
print("Kunci publik =", (e, n))
print("Kunci privat =", (d, n))
print("Verifikasi e*d mod phi(n) =", (e * d) % phi)


p = 5
q = 11
n = 55
phi(n) = 40
e = 3
d = 27
Kunci publik = (3, 55)
Kunci privat = (27, 55)
Verifikasi e*d mod phi(n) = 1


**Penjelasan:** Kunci publik terdiri dari `(e, n)` dan kunci privat terdiri dari `(d, n)`. Hasil verifikasi harus bernilai `1`, karena `e × d ≡ 1 (mod φ(n))`.

### 2. Enkripsi Pesan Numerik

Pesan numerik harus memenuhi `0 ≤ M < n`. Enkripsi dilakukan dengan rumus `C = M^e mod n`.


In [4]:
p, q, e = 5, 11, 3
n, phi, d = buat_kunci_rsa(p, q, e)

M = 7
if not 0 <= M < n:
    raise ValueError(f"Pesan harus memenuhi 0 <= M < {n}")

C = pow(M, e, n)

print("Pesan asli (M):", M)
print("Kunci publik (e, n):", (e, n))
print("Ciphertext (C):", C)


Pesan asli (M): 7
Kunci publik (e, n): (3, 55)
Ciphertext (C): 13


**Penjelasan:** Fungsi `pow(M, e, n)` menghitung perpangkatan sekaligus modulo dengan lebih efisien daripada menghitung `M ** e` terlebih dahulu.

### 3. Dekripsi Ciphertext

Ciphertext dari contoh sebelumnya didekripsi menggunakan kunci privat dengan rumus `M = C^d mod n`.


In [5]:
# Parameter dan kunci yang sama dengan bagian sebelumnya
p, q, e = 5, 11, 3
n, phi, d = buat_kunci_rsa(p, q, e)

C = 13
M_hasil = pow(C, d, n)

print("Ciphertext (C):", C)
print("Kunci privat (d, n):", (d, n))
print("Pesan hasil dekripsi (M):", M_hasil)


Ciphertext (C): 13
Kunci privat (d, n): (27, 55)
Pesan hasil dekripsi (M): 7


**Penjelasan:** Dengan parameter pada jobsheet, ciphertext `13` akan dikembalikan menjadi pesan asli `7`.

### 4. Enkripsi dan Dekripsi Beberapa Pesan

Beberapa nilai pesan diproses menggunakan pasangan kunci yang sama. Hasil dekripsi dibandingkan dengan daftar pesan asli.


In [6]:
p, q, e = 5, 11, 3
n, phi, d = buat_kunci_rsa(p, q, e)

pesan = [7, 10, 20, 3, 25]

ciphertext = []
for M in pesan:
    if not 0 <= M < n:
        raise ValueError(f"Pesan {M} di luar rentang 0 sampai {n-1}")
    ciphertext.append(pow(M, e, n))

hasil_dekripsi = [pow(C, d, n) for C in ciphertext]

print("Pesan asli:", pesan)
print("Ciphertext:", ciphertext)
print("Hasil dekripsi:", hasil_dekripsi)
print("Pesan berhasil dipulihkan:", pesan == hasil_dekripsi)


Pesan asli: [7, 10, 20, 3, 25]
Ciphertext: [13, 10, 25, 27, 5]
Hasil dekripsi: [7, 10, 20, 3, 25]
Pesan berhasil dipulihkan: True


**Penjelasan:** Setiap pesan dienkripsi satu per satu. Jika proses benar, daftar hasil dekripsi sama dengan daftar pesan asli.

### 5. Perpangkatan Modular dengan Repeated Squaring

Repeated squaring menghitung perpangkatan modular dengan memecah eksponen menjadi bentuk biner dan melakukan kuadrat berulang.


In [7]:
def repeated_squaring(base, exponent, modulus):
    """Menghitung (base ** exponent) mod modulus dengan repeated squaring."""
    if exponent < 0:
        raise ValueError("Eksponen harus bilangan bulat nonnegatif.")
    if modulus <= 0:
        raise ValueError("Modulus harus lebih besar dari nol.")

    hasil = 1
    base = base % modulus
    langkah = []

    while exponent > 0:
        if exponent % 2 == 1:
            hasil = (hasil * base) % modulus
        langkah.append((exponent, base, hasil))
        base = (base * base) % modulus
        exponent //= 2

    return hasil, langkah


# Parameter sesuai contoh RSA pada jobsheet
p, q, e = 5, 11, 3
n, phi, d = buat_kunci_rsa(p, q, e)
M = 7

hasil_manual, detail = repeated_squaring(M, e, n)
hasil_python = pow(M, e, n)

print("Ciphertext repeated squaring:", hasil_manual)
print("Ciphertext menggunakan pow():", hasil_python)
print("Hasil sama:", hasil_manual == hasil_python)


Ciphertext repeated squaring: 13
Ciphertext menggunakan pow(): 13
Hasil sama: True


**Penjelasan:** Hasil `repeated_squaring` dibandingkan dengan fungsi bawaan Python `pow()` untuk memastikan perhitungannya benar.

### 6. Enkripsi Kalimat dengan RSA-OAEP

RSA-OAEP adalah skema padding untuk enkripsi RSA. Pustaka `cryptography` digunakan agar proses enkripsi kalimat dilakukan dengan padding OAEP dan SHA-256.


In [ ]:
%pip install cryptography

In [10]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes

# Membuat pasangan kunci RSA
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)
public_key = private_key.public_key()

kalimat = "Saya sedang mempelajari algoritma RSA."
data = kalimat.encode("utf-8")

ciphertext = public_key.encrypt(
    data,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

hasil_dekripsi = private_key.decrypt(
    ciphertext,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
).decode("utf-8")

print("Kalimat asli:", kalimat)
print("Ciphertext (hex):", ciphertext.hex())
print("Hasil dekripsi:", hasil_dekripsi)
print("Berhasil:", hasil_dekripsi == kalimat)


Kalimat asli: Saya sedang mempelajari algoritma RSA.
Ciphertext (hex): 490eb5c3e05c0e102b57ecac31d4b45a497c061ab1d3326ba7fafdee6044aa87293fa4c54b136c9b1c67fa3225a4b32a228fb936432ebdf83b7bce65fa6825dee0b6545609e3458df4b55c6a328f7f23e79a08e550af928386c5f0eee82019b5f01f95038e4a1919013822947d11a4d03e1fff43772a9158cf9051d7a11a4857f978467a52df9c5086efb8bfd275d83b288e93af676c40e73a35f943d61d45d1e76b4be32bd4f8a82e7af68a5d673728381611b97c4c9905015a62d328e08d1a385b761da966cc7acf85b32b3a974c2960ab6daa1f070d4f90bc0f6c40f3e6902b64b25ecf0ccf2b4b7e156d788ca8261f4c965da3317134e23eb31330ae1770
Hasil dekripsi: Saya sedang mempelajari algoritma RSA.
Berhasil: True


**Penjelasan:** Pesan diubah menjadi byte, dienkripsi dengan kunci publik, lalu didekripsi dengan kunci privat. OAEP memiliki batas panjang pesan; untuk RSA 2048-bit dengan SHA-256, panjang pesan maksimum sekitar 190 byte.

### 7. RSA-AES (Hybrid Encryption)

Pada metode hybrid, AES digunakan untuk mengenkripsi data, sedangkan RSA-OAEP digunakan untuk mengenkripsi (membungkus) kunci AES. Cara ini cocok untuk data yang lebih panjang dibandingkan mengenkripsi seluruh data langsung dengan RSA.


In [11]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os

# Membuat pasangan kunci RSA
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)
public_key = private_key.public_key()

kalimat = "Komunikasi data dapat dilindungi menggunakan kriptografi."
data = kalimat.encode("utf-8")

# Membuat kunci AES dan nonce, kemudian mengenkripsi data
aes_key = AESGCM.generate_key(bit_length=256)
aesgcm = AESGCM(aes_key)
nonce = os.urandom(12)
ciphertext_aes = aesgcm.encrypt(nonce, data, None)

# Mengenkripsi kunci AES menggunakan RSA-OAEP
encrypted_aes_key = public_key.encrypt(
    aes_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Mendekripsi kunci AES menggunakan RSA privat
decrypted_aes_key = private_key.decrypt(
    encrypted_aes_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Mendekripsi data menggunakan kunci AES yang telah dipulihkan
hasil_dekripsi = AESGCM(decrypted_aes_key).decrypt(
    nonce, ciphertext_aes, None
).decode("utf-8")

print("Kalimat asli:", kalimat)
print("Ciphertext AES (hex):", ciphertext_aes.hex())
print("Kunci AES terenkripsi RSA (hex):", encrypted_aes_key.hex())
print("Hasil dekripsi:", hasil_dekripsi)
print("Berhasil:", hasil_dekripsi == kalimat)


Kalimat asli: Komunikasi data dapat dilindungi menggunakan kriptografi.
Ciphertext AES (hex): 14182944f5e32686dc95d85275b625d1ea2c31ed25f0a96c8f510ed78e8c4063472c84f236667f5f53a292d87d504ca61759f9519b4a4e50d74aec0a34a466141388e4b204ef98dfcc
Kunci AES terenkripsi RSA (hex): 3919ac283396a86744a7cfeab5afc5abb9a57a8a310fbae27264c771dc29cd996fe19981032603ac0b81829da909403445f35ed3ade14a72e7e04fcf26d9c69278dc66954157e948c40dbbb5f97cec68935ac6a108641c1fa670c166a6b11b898aece8e0df856e68086c66ac3851df7409ad9162f6721ca542f9dcca827593392aa2f0cbf8006bfab45b8fe50ab29bd51bad92e021c447d6eea9e8ba1ff400aee36fed816a1d09c512a0b4d6d831a974511818ddc4081bbf99b54a3b24f93fee166b86e5d4209b6de53f85d63898e4cb422b5074173743034ba844bf476421e739dcb6a8cfb499ca3a70bc29463767cfe21c9b3170ff449a080d5dff5bfb31b9
Hasil dekripsi: Komunikasi data dapat dilindungi menggunakan kriptografi.
Berhasil: True


## E. Evaluasi

### 1. Pembentukan kunci dengan p = 7, q = 17, dan e = 5

Program menampilkan `n`, φ(n), `d`, kunci publik, dan kunci privat. Program juga memeriksa syarat FPB dan invers modular.


In [12]:
from math import gcd

p, q, e = 7, 17, 5
n = p * q
phi = (p - 1) * (q - 1)

fpb = gcd(e, phi)
assert fpb == 1, "e tidak relatif prima terhadap phi(n)"

d = pow(e, -1, phi)
verifikasi = (e * d) % phi

print("p =", p)
print("q =", q)
print("n =", n)
print("phi(n) =", phi)
print("gcd(e, phi(n)) =", fpb)
print("d =", d)
print("Kunci publik =", (e, n))
print("Kunci privat =", (d, n))
print("(e*d) mod phi(n) =", verifikasi)
print("Pemeriksaan gcd berhasil:", fpb == 1)
print("Pemeriksaan invers modular berhasil:", verifikasi == 1)


p = 7
q = 17
n = 119
phi(n) = 96
gcd(e, phi(n)) = 1
d = 77
Kunci publik = (5, 119)
Kunci privat = (77, 119)
(e*d) mod phi(n) = 1
Pemeriksaan gcd berhasil: True
Pemeriksaan invers modular berhasil: True


**Penjelasan:** Diperoleh `n = 119`, `φ(n) = 96`, dan `d = 77`. Kedua pemeriksaan otomatis bernilai benar.

### 2. Program input pesan numerik dengan p = 11, q = 17, dan e = 3

Program meminta pengguna memasukkan pesan `M`, memvalidasi rentang `0 ≤ M < n`, lalu melakukan enkripsi dan dekripsi. Sel pengujian berikut menjalankan tiga nilai pesan berbeda.


In [14]:
p, q, e = 11, 17, 3
n, phi, d = buat_kunci_rsa(p, q, e)

def proses_pesan_numerik(M):
    if not isinstance(M, int) or not 0 <= M < n:
        raise ValueError(f"M harus bilangan bulat dengan 0 <= M < {n}")

    C = pow(M, e, n)
    hasil = pow(C, d, n)

    print("Pesan asli (M):", M)
    print("Ciphertext (C):", C)
    print("Hasil dekripsi:", hasil)
    print("Berhasil dipulihkan:", hasil == M)
    print("-" * 35)

print("n =", n, "| phi(n) =", phi, "| d =", d)
# Masukkan nilai M ketika sel dijalankan:
M_input = int(input(f"Masukkan pesan numerik M (0 sampai {n-1}): "))
proses_pesan_numerik(M_input)


n = 187 | phi(n) = 160 | d = 107
Pesan asli (M): 101
Ciphertext (C): 118
Hasil dekripsi: 101
Berhasil dipulihkan: True
-----------------------------------


In [15]:
# Pengujian minimal tiga nilai M yang berbeda
nilai_uji = [7, 50, 150]

for nilai in nilai_uji:
    proses_pesan_numerik(nilai)


Pesan asli (M): 7
Ciphertext (C): 156
Hasil dekripsi: 7
Berhasil dipulihkan: True
-----------------------------------
Pesan asli (M): 50
Ciphertext (C): 84
Hasil dekripsi: 50
Berhasil dipulihkan: True
-----------------------------------
Pesan asli (M): 150
Ciphertext (C): 24
Hasil dekripsi: 150
Berhasil dipulihkan: True
-----------------------------------


**Penjelasan:** Untuk parameter ini, `n = 187`. Nilai uji harus berada pada rentang 0 sampai 186. Tiga nilai pada sel pengujian digunakan untuk menunjukkan bahwa pesan dapat dipulihkan.

### 3. Enkripsi kalimat input pengguna menggunakan RSA-OAEP

Kalimat dimasukkan saat program berjalan, bukan ditetapkan sebagai isi pesan di dalam program. Hasil ciphertext ditampilkan dalam format hexadecimal.


In [16]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes

# Membuat pasangan kunci RSA
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)
public_key = private_key.public_key()

# Input kalimat dari pengguna
kalimat_input = input("Masukkan kalimat yang akan dienkripsi: ")
data = kalimat_input.encode("utf-8")

# Enkripsi menggunakan kunci publik RSA-OAEP
oaep = padding.OAEP(
    mgf=padding.MGF1(algorithm=hashes.SHA256()),
    algorithm=hashes.SHA256(),
    label=None
)

try:
    ciphertext = public_key.encrypt(data, oaep)

    # Dekripsi menggunakan kunci privat
    hasil_dekripsi = private_key.decrypt(ciphertext, oaep).decode("utf-8")

    print("\nKalimat asli:", kalimat_input)
    print("Ciphertext (hex):", ciphertext.hex())
    print("Hasil dekripsi:", hasil_dekripsi)
    print("Status kesamaan:", hasil_dekripsi == kalimat_input)

except ValueError as error:
    print("Pesan tidak dapat dienkripsi. Periksa panjang kalimat.")
    print("Detail:", error)



Kalimat asli: Bima Satria
Ciphertext (hex): 8d85b5b8e2052ba33a08a1943a29a64b2da0797711a4a6c4a043c2e817f78b7fc28303fcc20389cd1ca390e84ccb1aa7f9a6cb516911a169675b29da8807c3b7bb74643b1ed31704592da7ae90f0f46a97ec62defe387c97c5d3fd1ebcc19c8f9f9ec0b323365104fbe752d8b68703762c18312f093c1ae8ebf8d4f0e217fc4b903364979e20dd777e2005c6fa493867b857cd08d9eb88ba6abf42b641b4794d81090ea4fa47738b4e03ea918110f160d36cf242372062b191d3ed59b29ede8ff7ca0a5aa5390bf3fa8e46e5e43683fb1bece64a65521ffc804b2ed293ccf580b1fd9fe0c5d3e4c0ed358f1bbfee40432e82548f0fb84e6cf4036f136dbf359b
Hasil dekripsi: Bima Satria
Status kesamaan: True


**Penjelasan:** Enkripsi dilakukan menggunakan kunci publik RSA-OAEP dan dekripsi menggunakan kunci privat. Status kesamaan bernilai `True` jika kalimat hasil dekripsi sama dengan input. RSA-OAEP memiliki batas panjang pesan, sehingga untuk pesan panjang lebih tepat menggunakan pendekatan hybrid RSA-AES.

---

## Kesimpulan

1. RSA menggunakan pasangan kunci publik dan privat yang saling berkaitan.
2. Enkripsi numerik menggunakan `C = M^e mod n`, sedangkan dekripsi menggunakan `M = C^d mod n`.
3. Nilai pesan numerik harus berada pada rentang `0 ≤ M < n`.
4. Repeated squaring dapat digunakan untuk menghitung perpangkatan modular secara efisien.
5. RSA-OAEP menambahkan padding yang aman untuk enkripsi RSA.
6. Kombinasi RSA-AES memanfaatkan RSA untuk melindungi kunci AES dan AES untuk mengenkripsi data.

**Cara menjalankan:** Jalankan sel dari atas ke bawah (*Run All*). Jika `cryptography` belum tersedia, instal melalui sel persiapan terlebih dahulu.
